# Data.org Financial Health Prediction Challenge!

## Empowering Financial Inclusion Across Africa

### Unzip Dataset

In [4]:
!ls

 LICENSE
 dataorg-financial-health-prediction-challenge.zip
'dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n (1).zip:Zone.Identifier'
 financial-health-zindi-gbms.ipynb


In [5]:
!unzip dataorg-financial-health-prediction-challenge.zip

Archive:  dataorg-financial-health-prediction-challenge.zip
  inflating: manifest-c388a32489998f91d25543edfbe8b78f20251204-19827-1nvty2o.json  
  inflating: Test.csv                
  inflating: Train.csv               
  inflating: SampleSubmission.csv    
  inflating: VariableDefinitions.csv  
  inflating: Starter Notebook.ipynb  


## Library Import

In [72]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from numpy import random
from tqdm import tqdm
from pathlib import Path

from fastai.tabular.all import *
from ipywidgets import interact

from fastai.imports import *
np.set_printoptions(linewidth=130)
import gc

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import f1_score

import xgboost as xgb
from xgboost import plot_importance

import lightgbm as lgb
import catboost as cat

In [69]:
# GPU Availability Check for XGBoost
print("="*70)
print("GPU AVAILABILITY CHECK")
print("="*70)

import subprocess
import sys

# Check if CUDA is available
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("✅ NVIDIA GPU detected!")
        print("\nGPU Information:")
        print("-" * 70)
        # Parse nvidia-smi output for GPU name and memory
        for line in result.stdout.split('\n'):
            if 'NVIDIA' in line or 'CUDA' in line:
                print(line.strip())
        print("-" * 70)
        
        # Check XGBoost GPU support
        try:
            import xgboost as xgb
            # Try to create a simple GPU-based model
            test_params = {'device': 'cuda', 'tree_method': 'hist'}
            test_model = xgb.XGBClassifier(**test_params)
            print("\n✅ XGBoost GPU support is AVAILABLE!")
            print("   You can use: device='cuda', tree_method='hist'")
            gpu_available = True
        except Exception as e:
            print(f"\n⚠️  XGBoost installed but GPU support not working: {e}")
            print("   You may need to reinstall XGBoost with GPU support:")
            print("   pip install xgboost --upgrade")
            gpu_available = False
    else:
        print("❌ No NVIDIA GPU detected")
        gpu_available = False
except FileNotFoundError:
    print("❌ nvidia-smi not found - No NVIDIA GPU available")
    print("   Running on CPU only")
    gpu_available = False
except subprocess.TimeoutExpired:
    print("⚠️  nvidia-smi timeout - GPU may not be properly configured")
    gpu_available = False
except Exception as e:
    print(f"⚠️  Error checking GPU: {e}")
    gpu_available = False

# Set device based on availability
if gpu_available:
    DEVICE = 'cuda'
    print(f"\n🚀 Device set to: {DEVICE} (GPU acceleration enabled)")
else:
    DEVICE = 'cpu'
    print(f"\n💻 Device set to: {DEVICE} (CPU only)")

print("="*70)
print(f"\nUse this in your XGBoost parameters: 'device': '{DEVICE}'")
print("="*70)

GPU AVAILABILITY CHECK
❌ nvidia-smi not found - No NVIDIA GPU available
   Running on CPU only

💻 Device set to: cpu (CPU only)

Use this in your XGBoost parameters: 'device': 'cpu'


## Load Dataset

In [2]:
path = Path('')
path

Path('.')

In [3]:
!ls

 LICENSE
 SampleSubmission.csv
'Starter Notebook.ipynb'
 Test.csv
 Train.csv
 VariableDefinitions.csv
 dataorg-financial-health-prediction-challenge.zip
'dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n (1).zip:Zone.Identifier'
 financial-health-zindi-gbms.ipynb
 manifest-c388a32489998f91d25543edfbe8b78f20251204-19827-1nvty2o.json
 submission_fastai.csv


In [4]:
train_df = pd.read_csv(path/'Train.csv',index_col='ID')
test_df = pd.read_csv(path/'Test.csv',index_col='ID')
sub_df = pd.read_csv(path/'SampleSubmission.csv')
vd_df = pd.read_csv(path/'VariableDefinitions.csv')

## EDA

In [5]:
train_df.head()

,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,...,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target
ID,,,,,,,,,,,,,,,,,,,,,
ID_3CFL0U,eswatini,63.0,Yes,No,No,No,Yes,3000.0,6000.0,7000.0,...,Never had,Never had,NaN,6.0,Never had,Used to have but don’t have now,NaN,Never had,Never had,Low
ID_XWI7G3,zimbabwe,39.0,No,Yes,Yes,No,Yes,NaN,NaN,NaN,...,NaN,NaN,No,3.0,Never had,Never had,NaN,NaN,NaN,Medium
ID_TY93LV,malawi,34.0,Don’t know or N/A,No,No,Don't know,Yes,30000.0,6000.0,13000.0,...,Never had,Never had,Yes,NaN,NaN,NaN,Yes,NaN,NaN,Low
ID_9OP2C8,malawi,28.0,Yes,No,No,No,No,180000.0,60000.0,30000.0,...,Never had,Never had,No,NaN,NaN,NaN,Yes,Never had,Have now,Low
ID_13REYS,zimbabwe,43.0,Yes,No,No,Yes,Yes,50.0,2400.0,1800.0,...,NaN,NaN,No,0.0,Never had,Never had,Yes,NaN,NaN,Low


In [6]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
owner_age,9618.0,4.170534e+01,1.331401e+01,18.0,32.0,40.0,50.0,103.0
personal_income,9509.0,2.627345e+05,2.566268e+06,0.0,300.0,2000.0,25000.0,150000000.0
business_expenses,9389.0,4.583838e+05,6.184746e+06,0.0,700.0,3000.0,25000.0,500000000.0
business_turnover,9402.0,1.348210e+06,8.804741e+06,0.0,1500.0,6000.0,50000.0,420000000.0
business_age_years,9366.0,7.030536e+00,7.650349e+00,0.0,2.0,4.0,10.0,60.0
business_age_months,5507.0,3.636281e+00,3.386488e+00,0.0,0.0,3.0,6.0,11.0


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9618 entries, ID_3CFL0U to ID_4XGOHM
Data columns (total 38 columns):
 #   Column                                                            Non-Null Count  Dtype  
---  ------                                                            --------------  -----  
 0   country                                                           9618 non-null   object 
 1   owner_age                                                         9618 non-null   float64
 2   attitude_stable_business_environment                              9616 non-null   object 
 3   attitude_worried_shutdown                                         9616 non-null   object 
 4   compliance_income_tax                                             9614 non-null   object 
 5   perception_insurance_doesnt_cover_losses                          9613 non-null   object 
 6   perception_cannot_afford_insurance                                9613 non-null   object 
 7   personal_income          

In [8]:
missing = pd.DataFrame({
    'count': train_df.isnull().sum(),
    'pct': (train_df.isnull().sum() / len(train_df) * 100).round(2)
})
print(missing[missing['count'] > 0])

                                                                  count    pct
attitude_stable_business_environment                                  2   0.02
attitude_worried_shutdown                                             2   0.02
compliance_income_tax                                                 4   0.04
perception_insurance_doesnt_cover_losses                              5   0.05
perception_cannot_afford_insurance                                    5   0.05
personal_income                                                     109   1.13
business_expenses                                                   229   2.38
business_turnover                                                   216   2.25
business_age_years                                                  252   2.62
motor_vehicle_insurance                                            2244  23.33
has_mobile_money                                                   2751  28.60
current_problem_cash_flow                           

In [9]:
missing = pd.DataFrame({
    'count': test_df.isnull().sum(),
    'pct': (test_df.isnull().sum() / len(train_df) * 100).round(2)
})
print(missing[missing['count'] > 0])

                                                                  count    pct
owner_age                                                             1   0.01
perception_insurance_doesnt_cover_losses                              2   0.02
perception_cannot_afford_insurance                                    2   0.02
personal_income                                                      23   0.24
business_expenses                                                    70   0.73
business_turnover                                                    70   0.73
business_age_years                                                   59   0.61
motor_vehicle_insurance                                             557   5.79
has_mobile_money                                                    710   7.38
current_problem_cash_flow                                           932   9.69
has_cellphone                                                       486   5.05
owner_sex                                           

In [10]:
test_df

,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,...,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender
ID,,,,,,,,,,,,,,,,,,,,,
ID_5EGLKX,zimbabwe,50.0,No,No,No,No,Yes,100.0,3600.0,7200.0,...,NaN,NaN,NaN,No,8.0,Never had,Never had,NaN,NaN,NaN
ID_4AI7RE,lesotho,36.0,Yes,Yes,No,Yes,Yes,900.0,400.0,900.0,...,NaN,NaN,NaN,Yes,NaN,NaN,NaN,Yes,Used to have but don't have now,Used to have but don't have now
ID_V9OB3M,lesotho,25.0,Don’t know or N/A,No,No,Don't know,Don't know,5250.0,350.0,1000.0,...,Used to have but don't have now,Have now,Have now,Yes,NaN,NaN,NaN,No,Never had,Used to have but don't have now
ID_6OI9DI,malawi,25.0,Don’t know or N/A,Yes,No,No,Yes,485000.0,10000.0,20000.0,...,Never had,Never had,Never had,Yes,NaN,NaN,NaN,Yes,Have now,Never had
ID_H2TN8B,lesotho,47.0,No,Yes,No,Don't know,Don't know,97.0,500.0,2000.0,...,Used to have but don't have now,Have now,Have now,Yes,NaN,NaN,NaN,Yes,Used to have but don't have now,Used to have but don't have now
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ID_FX7XJZ,eswatini,29.0,Yes,Yes,No,No,Yes,600.0,1700.0,2000.0,...,Never had,Never had,Never had,NaN,11.0,Never had,Never had,NaN,Never had,Never had
ID_XAL1LX,malawi,20.0,Don’t know or N/A,Don’t know or N/A,No,Don't know,Don't know,30000.0,20000.0,25000.0,...,Never had,Never had,Never had,No,4.0,NaN,NaN,Yes,NaN,NaN
ID_UHBP0F,zimbabwe,26.0,Yes,Yes,No,Yes,Yes,3888.0,NaN,NaN,...,NaN,NaN,NaN,No,0.0,Have now,Have now,NaN,NaN,NaN


In [11]:
# Number of unique categories
train_df['Target'].nunique()

# Count per category
train_df['Target'].value_counts()

Target
Low       6280
Medium    2868
High       470
Name: count, dtype: int64

## Prepare data for Machine Learning

In [12]:
#splits = RandomSplitter(valid_pct=0.2)(range_of(original_df))
#train_df = pd.concat([train_df, original_df], ignore_index=True) *

cont_names,cat_names = cont_cat_split(train_df, dep_var='Target')

In [13]:
splits = RandomSplitter(valid_pct=0.2)(range_of(train_df))

In [14]:
to = TabularPandas(train_df, procs=[Categorify, FillMissing,Normalize],
                   cat_names = cat_names,
                   cont_names = cont_names,
                   y_names='Target',
                   y_block=CategoryBlock(),
                   splits=splits)

/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)
/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [15]:
cont_names

['owner_age',
 'personal_income',
 'business_expenses',
 'business_turnover',
 'business_age_years',
 'business_age_months']

In [16]:
dls = to.dataloaders(bs=64)

In [17]:
dls.show_batch()

,country,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,owner_sex,offers_credit_to_customers,attitude_satisfied_with_achievement,has_credit_card,keeps_financial_records,perception_insurance_companies_dont_insure_businesses_like_yours,perception_insurance_important,has_insurance,covid_essential_service,attitude_more_successful_next_year,problem_sourcing_money,marketing_word_of_mouth,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,personal_income_na,business_expenses_na,business_turnover_na,business_age_years_na,business_age_months_na,owner_age,personal_income,business_expenses,business_turnover,business_age_years,business_age_months,Target
0,zimbabwe,Yes,Yes,No,Don't know,Don't know,Never had,#na#,Yes,No,Female,No,Yes,Never had,"Yes, sometimes",Don't know,Do not know / N‎/A,No,No,#na#,#na#,#na#,#na#,#na#,#na#,Yes,Never had,Never had,Yes,#na#,#na#,False,False,False,False,False,24.999999,3.699924e+02,2400.000685,3.599950e+03,6.0,1.416142e-07,Low
1,malawi,Don’t know or N/A,Yes,No,Don't know,Yes,#na#,Never had,No,No,Female,"Yes, always",Yes,Never had,No,Yes,No,No,#na#,Yes,Yes,No,Never had,Never had,Never had,No,#na#,#na#,No,#na#,#na#,False,False,False,False,True,42.000000,1.080000e+06,539999.998932,1.080000e+06,11.0,3.000000e+00,Low
2,malawi,Don’t know or N/A,Yes,No,Yes,Yes,#na#,Have now,Yes,Yes,Male,"Yes, sometimes",No,Never had,Yes,Don?t know / doesn?t apply,Yes,No,#na#,Don't know,Yes,No,Never had,Never had,Never had,No,#na#,#na#,Yes,#na#,#na#,False,False,False,False,True,26.000000,5.000000e+04,30000.007455,2.999994e+04,5.0,3.000000e+00,Low
3,lesotho,Yes,No,No,No,Yes,Never had,Have now,Yes,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,No,Yes,No,Yes,Used to have but don't have now,Have now,Have now,#na#,#na#,#na#,No,Used to have but don't have now,Used to have but don't have now,False,False,False,False,True,28.000000,1.999999e+03,2999.984538,5.000037e+03,10.0,3.000000e+00,Low
4,lesotho,Yes,Yes,No,Yes,Yes,Never had,Have now,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,Don't know,Yes,No,Yes,Used to have but don't have now,Never had,Never had,#na#,#na#,#na#,Yes,Used to have but don't have now,Used to have but don't have now,False,False,False,False,True,44.000000,7.000003e+03,199.993266,9.999794e+02,1.0,3.000000e+00,Low
5,malawi,No,No,No,Don't know,Don't know,#na#,Have now,No,Yes,Female,"Yes, sometimes",No,Never had,No,Don?t know / doesn?t apply,Don?t know / doesn?t apply,No,#na#,Yes,No,No,Never had,Never had,Never had,No,#na#,#na#,Yes,#na#,#na#,False,False,False,True,False,32.000000,5.200000e+05,45000.003530,6.260000e+06,4.0,8.000000e+00,Low
6,eswatini,No,No,No,No,Yes,Never had,Have now,Yes,Yes,Male,"Yes, always",No,Never had,No,No,Yes,No,No,Yes,Yes,Yes,Never had,Never had,Never had,#na#,Never had,Used to have but don’t have now,#na#,Never had,Never had,False,False,False,False,False,58.000001,1.000006e+03,1300.021960,2.499986e+03,2.0,6.000000e+00,Low
7,lesotho,Yes,No,No,Don't know,Yes,Never had,Have now,Yes,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,#na#,Don't know,Yes,Yes,Yes,#na#,#na#,#na#,#na#,#na#,#na#,No,Used to have but don't have now,Used to have but don't have now,False,False,False,False,True,39.000000,4.999992e+02,100.004285,5.000177e+02,2.0,3.000000e+00,Medium
8,malawi,Don’t know or N/A,No,Yes,No,Yes,#na#,Never had,Yes,Yes,Female,No,No,Have now,Yes,No,Don?t know / doesn?t apply,No,#na#,Yes,No,Yes,Never had,Never had,Have now,No,#na#,#na#,No,#na#,#na#,False,False,False,False,True,36.000000,1.455000e+04,312000.001126,3.145500e+05,4.0,3.000000e+00,Medium
9,zimbabwe,Yes,Don’t know or N/A,No,No,Yes,Never had,Have now,Yes,Yes,Male,No,Yes,Never had,No,No,Yes,No,No,#na#,#na#,#na#,#na#,#na#,#na#,No,Nev

In [18]:
learn = tabular_learner(dls, metrics=F1ScoreMulti(average='macro'))

In [19]:
learn = tabular_learner(dls, metrics=F1Score(average='macro'))

In [20]:
learn.fit_one_cycle(50)

epoch,train_loss,valid_loss,f1_score,time
0,1.085382,1.074330,0.383792,00:01
1,1.019640,0.966779,0.453516,00:01
2,0.815484,0.712001,0.530218,00:01
3,0.571171,0.606600,0.659943,00:01
4,0.459812,0.483293,0.757276,00:01
5,0.399505,0.471081,0.757039,00:01
6,0.373385,0.411680,0.799205,00:01
7,0.347767,0.371722,0.806084,00:01
8,0.319367,0.429453,0.783650,00:01
9,0.302277,0.401387,0.804695,00:01


In [21]:
learn.show_results()

,country,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,owner_sex,offers_credit_to_customers,attitude_satisfied_with_achievement,has_credit_card,keeps_financial_records,perception_insurance_companies_dont_insure_businesses_like_yours,perception_insurance_important,has_insurance,covid_essential_service,attitude_more_successful_next_year,problem_sourcing_money,marketing_word_of_mouth,has_loan_account,has_internet_banking,has_debit_card,future_risk_theft_stock,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,personal_income_na,business_expenses_na,business_turnover_na,business_age_years_na,business_age_months_na,owner_age,personal_income,business_expenses,business_turnover,business_age_years,business_age_months,Target,Target_pred
0,1.0,3.0,2.0,4.0,2.0,3.0,3.0,2.0,1.0,2.0,1.0,3.0,4.0,3.0,4.0,4.0,5.0,1.0,2.0,4.0,2.0,2.0,4.0,4.0,3.0,0.0,4.0,2.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.834758,-0.097535,-0.067449,-0.160656,0.812915,0.636514,2.0,2.0
1,3.0,3.0,2.0,2.0,2.0,3.0,0.0,2.0,3.0,1.0,1.0,3.0,3.0,3.0,2.0,4.0,5.0,1.0,0.0,4.0,1.0,1.0,4.0,4.0,3.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,-0.426367,0.646925,1.870526,1.753525,-0.788224,-0.137258,1.0,1.0
2,1.0,3.0,2.0,4.0,2.0,3.0,3.0,2.0,1.0,2.0,2.0,3.0,4.0,3.0,3.0,4.0,5.0,1.0,3.0,4.0,2.0,2.0,4.0,3.0,2.0,0.0,4.0,3.0,0.0,4.0,6.0,1.0,1.0,1.0,1.0,1.0,-1.029333,-0.095486,-0.065869,-0.158692,-0.654796,2.184057,1.0,1.0
3,3.0,2.0,3.0,2.0,3.0,3.0,0.0,2.0,2.0,2.0,2.0,1.0,4.0,3.0,2.0,4.0,4.0,1.0,0.0,3.0,1.0,2.0,4.0,4.0,4.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,0.779566,0.721427,-0.064825,0.510829,3.348052,-0.137258,1.0,1.0
4,1.0,2.0,3.0,2.0,2.0,3.0,3.0,2.0,1.0,2.0,2.0,3.0,3.0,3.0,3.0,4.0,4.0,1.0,3.0,4.0,2.0,0.0,4.0,4.0,3.0,0.0,4.0,3.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,-0.577108,-0.097878,-0.065123,-0.158201,0.012346,2.957829,1.0,1.0
5,1.0,3.0,2.0,2.0,2.0,3.0,3.0,2.0,3.0,2.0,2.0,3.0,3.0,3.0,3.0,4.0,5.0,1.0,2.0,4.0,2.0,2.0,6.0,4.0,3.0,0.0,4.0,3.0,0.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.382533,-0.097051,-0.067539,-0.154372,-0.921652,1.797171,2.0,1.0
6,3.0,1.0,3.0,2.0,1.0,1.0,0.0,3.0,3.0,2.0,1.0,3.0,3.0,3.0,2.0,3.0,5.0,1.0,0.0,4.0,1.0,1.0,4.0,4.0,2.0,1.0,0.0,0.0,2.0,3.0,4.0,1.0,1.0,1.0,1.0,2.0,-0.426367,-0.079468,-0.066018,-0.158692,0.012346,-0.137258,1.0,1.0
7,3.0,1.0,3.0,4.0,1.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,3.0,3.0,2.0,3.0,5.0,1.0,0.0,4.0,1.0,1.0,4.0,4.0,3.0,2.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,-1.481558,-0.004967,-0.045442,0.148146,-0.788224,-0.137258,1.0,1.0
8,4.0,3.0,3.0,2.0,1.0,3.0,3.0,0.0,3.0,1.0,1.0,1.0,3.0,3.0,3.0,2.0,5.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,3.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,-0.803221,-0.098072,-0.067753,-0.161059,-0.788224,-1.297915,1.0,1.0


In [22]:
test_to = to.new(test_df)

In [23]:
test_processed = test_to.items.copy()
test_processed.shape, test_df.shape

((2405, 37), (2405, 37))

In [24]:
test_processed['owner_age'].fillna(test_processed['owner_age'].median(), inplace=True)

/tmp/ipykernel_66673/2900860375.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_processed['owner_age'].fillna(test_processed['owner_age'].median(), inplace=True)


In [25]:
test_dl = dls.test_dl(test_processed)
test_dl

/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)
/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [26]:
#dl = learn.dls.test_dl(test_dl)

In [27]:
dl = learn.dls.test_dl(test_processed)

/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  to[n].fillna(self.na_dict[n], inplace=True)
/home/rubanza/.local/lib/python3.10/site-packages/fastai/tabular/core.py:314: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [28]:
learn.get_preds(dl=dl)

(tensor([[1.4126e-04, 9.7848e-01, 2.1375e-02],
         [2.6323e-04, 6.9684e-01, 3.0290e-01],
         [2.9177e-06, 9.9996e-01, 4.1805e-05],
         ...,
         [2.2110e-07, 6.8395e-06, 9.9999e-01],
         [4.5709e-03, 2.9062e-05, 9.9540e-01],
         [2.6004e-06, 9.9999e-01, 1.1975e-05]]),
 None)

In [29]:
xc = learn.get_preds(dl=dl)
xc

(tensor([[1.4126e-04, 9.7848e-01, 2.1375e-02],
         [2.6323e-04, 6.9684e-01, 3.0290e-01],
         [2.9177e-06, 9.9996e-01, 4.1805e-05],
         ...,
         [2.2110e-07, 6.8395e-06, 9.9999e-01],
         [4.5709e-03, 2.9062e-05, 9.9540e-01],
         [2.6004e-06, 9.9999e-01, 1.1975e-05]]),
 None)

## Creating Submission File from FastAI Predictions

### What does `learn.get_preds(dl=dl)` do?

`learn.get_preds(dl=dl)` is a FastAI method that generates predictions from your trained neural network model:

**Returns a tuple:**
- **First element**: A tensor of predicted probabilities with shape `(n_samples, n_classes)`
  - In our case: `(2405, 3)` - 2405 test samples × 3 classes (High, Low, Medium)
  - Each row contains 3 probability values that sum to ~1.0
  - Example: `[0.00000171, 0.98901, 0.010984]` means:
    - 0.0002% probability of "High"
    - 98.9% probability of "Low" 
    - 1.1% probability of "Medium"
- **Second element**: `None` (would contain targets if they existed in the test set)

**How it works:**
1. Takes your test dataloader (`dl`) containing preprocessed test data
2. Passes each sample through your trained neural network
3. Applies softmax to get probability distributions for each class
4. Returns the probabilities for all test samples

### Creating a Submission File

To create a submission file, follow these steps:

1. **Get predictions**: Call `learn.get_preds(dl=dl)` to get probability tensor
2. **Find winning class**: Use `argmax()` to get the index of highest probability
3. **Map to labels**: Convert numeric indices (0, 1, 2) to class names using `to.vocab`
4. **Create DataFrame**: Build submission with ID and Target columns
5. **Save to CSV**: Export in the format expected by the competition

The code in the next cell demonstrates this complete workflow.

In [30]:
# Create submission file from FastAI neural network predictions

# Step 1: Get predictions (probabilities for each class)
preds, _ = learn.get_preds(dl=dl)

# Step 2: Convert probabilities to class indices (0, 1, 2)
# argmax gets the index of the highest probability
pred_classes = preds.argmax(dim=1)

# Step 3: Convert numeric indices to actual class names (High, Low, Medium)
# to.vocab contains the mapping from indices to class labels
pred_labels = [to.vocab[i] for i in pred_classes]

# Step 4: Create submission dataframe matching the sample submission format
submission = pd.DataFrame({
    'ID': test_df.index,  # IDs from test set
    'Target': pred_labels
})

# Step 5: Verify format matches sample submission
print(f"Submission shape: {submission.shape}")
print(f"Sample submission shape: {sub_df.shape}")
print(f"\nClass distribution in predictions:")
print(submission['Target'].value_counts())
print(f"\nFirst few predictions:")
print(submission.head())

# Step 6: Save to CSV
submission.to_csv('submission_fastai.csv', index=False)
print(f"\nSubmission saved to submission_fastai.csv")

Submission shape: (2405, 2)
Sample submission shape: (2405, 2)

Class distribution in predictions:
Target
Low       1586
Medium     717
High       102
Name: count, dtype: int64

First few predictions:
          ID Target
0  ID_5EGLKX    Low
1  ID_4AI7RE    Low
2  ID_V9OB3M    Low
3  ID_6OI9DI    Low
4  ID_H2TN8B    Low

Submission saved to submission_fastai.csv


In [31]:
len(xc)

2

In [32]:
xc[0].shape

torch.Size([2405, 3])

In [33]:
xcy = xc[0]
xcy

tensor([[1.4126e-04, 9.7848e-01, 2.1375e-02],
        [2.6323e-04, 6.9684e-01, 3.0290e-01],
        [2.9177e-06, 9.9996e-01, 4.1805e-05],
        ...,
        [2.2110e-07, 6.8395e-06, 9.9999e-01],
        [4.5709e-03, 2.9062e-05, 9.9540e-01],
        [2.6004e-06, 9.9999e-01, 1.1975e-05]])

In [76]:
sub_df

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium


In [35]:
X_train, y_train = to.train.xs, to.train.ys.values.ravel()
X_test, y_test = to.valid.xs, to.valid.ys.values.ravel()

In [36]:
X_train

,country,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,motor_vehicle_insurance,has_mobile_money,current_problem_cash_flow,has_cellphone,...,business_expenses_na,business_turnover_na,business_age_years_na,business_age_months_na,owner_age,personal_income,business_expenses,business_turnover,business_age_years,business_age_months
ID,,,,,,,,,,,,,,,,,,,,,
ID_J2XORV,1,3,3,4,2,3,3,3,1,1,...,1,1,1,1,-0.953963,-0.097349,-0.062589,-0.143964,1.213200,0.636514
ID_GOCRK5,2,3,1,1,3,3,3,0,3,0,...,1,1,1,2,-0.878592,-0.097535,-0.067509,-0.160533,-0.521367,-0.137258
ID_0BHNL9,4,3,2,2,2,3,3,0,0,2,...,1,1,1,1,0.251971,-0.098004,-0.067607,-0.161032,-0.788224,-1.297915
ID_CIFL79,2,3,2,2,2,2,1,2,3,0,...,1,1,1,2,-0.124884,-0.097945,-0.067750,-0.161098,1.346628,-0.137258
ID_6COXKQ,2,1,1,2,1,3,3,0,3,0,...,1,1,1,2,1.081050,-0.098058,-0.067793,-0.161135,3.081196,-0.137258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ID_JV9LHA,1,2,2,4,3,3,3,3,3,2,...,1,1,1,1,2.814578,-0.060843,-0.056028,-0.148874,4.415479,-1.297915
ID_9JP50E,3,2,2,2,1,1,0,2,2,2,...,1,1,1,2,0.553454,-0.004967,-0.049915,-0.137827,-0.254511,-0.137258
ID_J4BIOH,2,2,3,4,2,2,3,2,0,0,...,1,1,1,2,-0.953963,-0.086919,-0.065571,-0.158692,0.812915,-0.137258


## Modelling

In [47]:
# ==========================================
# XGBoost Default Parameters (Baseline)
# ==========================================
# These are the default values for XGBoost multiclass classification
# Use this as your baseline before hyperparameter tuning

xgb_default_params = {
    # ==========================================
    # TASK PARAMETERS (Required for multiclass)
    # ==========================================
    'objective': 'multi:softprob',      # Multiclass with probability output
    'num_class': 3,                     # 3 classes: Low, Medium, High
    'eval_metric': 'mlogloss',          # Multiclass log loss
    
    # ==========================================
    # TREE STRUCTURE (Core parameters)
    # ==========================================
    'max_depth': 6,                     # DEFAULT: 6 - Maximum tree depth
    'min_child_weight': 1,              # DEFAULT: 1 - Min sum of instance weight in child
    'gamma': 0,                         # DEFAULT: 0 - Min loss reduction for split
    
    # ==========================================
    # LEARNING RATE & BOOSTING
    # ==========================================
    'learning_rate': 0.3,               # DEFAULT: 0.3 (eta) - Step size shrinkage
    'n_estimators': 100,                # DEFAULT: 100 - Number of boosting rounds
    
    # ==========================================
    # SAMPLING (Prevents overfitting)
    # ==========================================
    'subsample': 1.0,                   # DEFAULT: 1.0 - Row sampling ratio
    'colsample_bytree': 1.0,            # DEFAULT: 1.0 - Column sampling per tree
    'colsample_bylevel': 1.0,           # DEFAULT: 1.0 - Column sampling per level
    'colsample_bynode': 1.0,            # DEFAULT: 1.0 - Column sampling per node
    
    # ==========================================
    # REGULARIZATION
    # ==========================================
    'reg_alpha': 0,                     # DEFAULT: 0 - L1 regularization (Lasso)
    'reg_lambda': 1,                    # DEFAULT: 1 - L2 regularization (Ridge)
    
    # ==========================================
    # CLASS IMBALANCE (Your case: High=4.9%)
    # ==========================================
    'max_delta_step': 0,                # DEFAULT: 0 - Max output change (0=no limit)
    # Note: For multiclass imbalance, use sample_weight in fit() instead of scale_pos_weight
    
    # ==========================================
    # SYSTEM PARAMETERS
    # ==========================================
    'tree_method': 'hist',              # Use 'hist' for speed (default: 'auto')
    'random_state': 42,                 # For reproducibility (default: 0)
    'n_jobs': -1,                       # Use all CPU cores (default: None)
    'verbosity': 0,                     # Silent mode (default: 1)
}

# Print summary
print("="*70)
print("XGBoost Default Parameters Loaded")
print("="*70)
print(f"Total parameters: {len(xgb_default_params)}")
print(f"\nKey settings:")
print(f"  Objective: {xgb_default_params['objective']}")
print(f"  Classes: {xgb_default_params['num_class']}")
print(f"  Max depth: {xgb_default_params['max_depth']}")
print(f"  Learning rate: {xgb_default_params['learning_rate']}")
print(f"  N estimators: {xgb_default_params['n_estimators']}")
print(f"  Tree method: {xgb_default_params['tree_method']}")
print("="*70)

# Train with default parameters using 5-Fold Stratified CV
print("\n" + "="*70)
print("Training XGBoost with DEFAULT parameters using 5-Fold Stratified CV...")
print("="*70)

# Setup cross-validation
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores_default = []

# Perform cross-validation
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold + 1}/5")
    
    # Split data for this fold
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create and train model
    model_fold = xgb.XGBClassifier(**xgb_default_params)
    model_fold.fit(X_tr, y_tr, verbose=False)
    
    # Predict and evaluate
    y_pred = model_fold.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores_default.append(f1)
    
    print(f"  Fold {fold + 1} F1 Macro: {f1:.6f}")

# Calculate mean and std
mean_f1_default = np.mean(f1_scores_default)
std_f1_default = np.std(f1_scores_default)

# Show results
print("\n" + "="*70)
print("RESULTS: Default Parameters with 5-Fold Cross-Validation")
print("="*70)
print(f"Mean F1 Macro Score: {mean_f1_default:.6f} (+/- {std_f1_default:.6f})")
print(f"\nFold scores: {[f'{score:.6f}' for score in f1_scores_default]}")
print(f"Min: {min(f1_scores_default):.6f}")
print(f"Max: {max(f1_scores_default):.6f}")
print("="*70)

# Train final model on full training data
print("\nTraining final model on full training data...")
model_default = xgb.XGBClassifier(**xgb_default_params)
model_default.fit(X_train, y_train, verbose=False)
print("✅ Final model trained!")
print("="*70)

XGBoost Default Parameters Loaded
Total parameters: 19

Key settings:
  Objective: multi:softprob
  Classes: 3
  Max depth: 6
  Learning rate: 0.3
  N estimators: 100
  Tree method: hist


In [55]:
# ==========================================
# XGBoost BETTER Starting Parameters
# ==========================================
# These are IMPROVED values (NOT defaults) for better baseline performance
# Expected improvement: F1 ~ 0.810-0.815 (vs default 0.8002)

xgb_better_params = {
    # ==========================================
    # TASK PARAMETERS (Same as defaults)
    # ==========================================
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    
    # ==========================================
    # TREE STRUCTURE (Keep conservative defaults)
    # ==========================================
    'max_depth': 6,                     # SAME - Good balance
    'min_child_weight': 1,              # SAME - Allow flexibility
    'gamma': 0,                         # SAME - No penalty initially
    
    # ==========================================
    # LEARNING RATE & BOOSTING (IMPROVED)
    # ==========================================
    'learning_rate': 0.1,               # CHANGED from 0.3 → Better generalization
    'n_estimators': 500,                # CHANGED from 100 → More learning capacity
    
    # ==========================================
    # SAMPLING (IMPROVED - Prevents overfitting)
    # ==========================================
    'subsample': 0.8,                   # CHANGED from 1.0 → Add row sampling
    'colsample_bytree': 0.8,            # CHANGED from 1.0 → Add column sampling
    'colsample_bylevel': 1.0,           # SAME - Not needed yet
    'colsample_bynode': 1.0,            # SAME - Not needed yet
    
    # ==========================================
    # REGULARIZATION (Keep defaults for now)
    # ==========================================
    'reg_alpha': 0,                     # SAME - No L1 yet
    'reg_lambda': 1,                    # SAME - Default L2
    
    # ==========================================
    # CLASS IMBALANCE (Same as defaults)
    # ==========================================
    'max_delta_step': 0,                # SAME - No capping yet
    
    # ==========================================
    # SYSTEM PARAMETERS
    # ==========================================
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

# Create Stratified K-Fold for cross-validation
from sklearn.model_selection import StratifiedKFold

print("Training XGBoost with BETTER parameters using 5-Fold Stratified CV...")
print("="*70)

# Setup cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores_better = []

# Perform cross-validation
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold + 1}/5")
    
    # Split data for this fold
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create and train model
    model_fold = xgb.XGBClassifier(**xgb_better_params)
    model_fold.fit(X_tr, y_tr, verbose=False)
    
    # Predict and evaluate
    y_pred = model_fold.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores_better.append(f1)
    
    print(f"  Fold {fold + 1} F1 Macro: {f1:.6f}")

# Calculate mean and std
mean_f1_better = np.mean(f1_scores_better)
std_f1_better = np.std(f1_scores_better)

# Show results
print("\n" + "="*70)
print("RESULTS: Better Parameters with 5-Fold Cross-Validation")
print("="*70)
print(f"Mean F1 Macro Score: {mean_f1_better:.6f} (+/- {std_f1_better:.6f})")
print(f"\nFold scores: {[f'{score:.6f}' for score in f1_scores_better]}")
print(f"Min: {min(f1_scores_better):.6f}")
print(f"Max: {max(f1_scores_better):.6f}")
print("="*70)

# Train final model on full training data for later use
print("\nTraining final model on full training data...")
model_better = xgb.XGBClassifier(**xgb_better_params)
model_better.fit(X_train, y_train, verbose=False)
print("✅ Final model trained!")
print("="*70)

# Show what changed from defaults
print("\nKey changes from defaults:")
print(f"  learning_rate:     0.3 → {xgb_better_params['learning_rate']}")
print(f"  n_estimators:      100 → {xgb_better_params['n_estimators']}")
print(f"  subsample:         1.0 → {xgb_better_params['subsample']}")
print(f"  colsample_bytree:  1.0 → {xgb_better_params['colsample_bytree']}")
print("="*70)

Training XGBoost with BETTER parameters (no sample weights)...
RESULTS: Better Starting Parameters
Validation F1 Macro Score: 0.812445
Baseline (default params):  0.8002
Improvement:               +0.012245
Percentage improvement:    +1.53%

✅ Better parameters improved the score!

Key changes from defaults:
  learning_rate:     0.3 → 0.1
  n_estimators:      100 → 500
  subsample:         1.0 → 0.8
  colsample_bytree:  1.0 → 0.8


In [56]:
xgb_better_params

{'objective': 'multi:softprob',
 'num_class': 3,
 'eval_metric': 'mlogloss',
 'max_depth': 6,
 'min_child_weight': 1,
 'gamma': 0,
 'learning_rate': 0.1,
 'n_estimators': 500,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'colsample_bylevel': 1.0,
 'colsample_bynode': 1.0,
 'reg_alpha': 0,
 'reg_lambda': 1,
 'max_delta_step': 0,
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1,
 'verbosity': 0}

In [ ]:
# Optuna Hyperparameter Tuning - Objective Function (Simplified)
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import xgboost as xgb

print("="*70)
print("OPTUNA HYPERPARAMETER TUNING SETUP")
print("="*70)

def objective(trial):
    """
    Optuna objective function for XGBoost hyperparameter tuning.
    Optimizes F1 Macro Score using 5-Fold Stratified Cross-Validation.
    """
    
    # Define hyperparameter search space
    params = {
        # GPU/CPU device
        'device': DEVICE if 'DEVICE' in globals() else 'cpu',
        'tree_method': 'hist',
        
        # Fixed parameters
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0,
        
        # Hyperparameters to tune
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
    }
    
    # Perform 5-fold stratified cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                  early_stopping_rounds=50, verbose=False)
        
        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred, average='macro')
        f1_scores.append(f1)
    
    return np.mean(f1_scores)

print("✅ Objective function defined!")
print("\nSearch space: 11 hyperparameters")
print("  max_depth, learning_rate, n_estimators,")
print("  min_child_weight, subsample, colsample_bytree,")
print("  colsample_bylevel, gamma, reg_alpha, reg_lambda, max_delta_step")
print("\nMetric: F1 Macro Score")
print("CV: 5-Fold Stratified Cross-Validation")
print("="*70)

In [ ]:
# Run Optuna Hyperparameter Optimization
import time
from datetime import datetime

print("="*70)
print("STARTING OPTUNA HYPERPARAMETER OPTIMIZATION")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device: {DEVICE if 'DEVICE' in globals() else 'cpu'}")
print("="*70)

# Create study
study = optuna.create_study(
    direction='maximize',
    study_name='xgboost_f1_optimization'
)

n_trials = 100
timeout = 7200  # 2 hours

print(f"\nRunning {n_trials} trials (max {timeout/3600:.1f} hours)...")
print("Progress will be shown every 10 trials.")
print("-"*70)

start_time = time.time()

def callback(study, trial):
    if trial.number % 10 == 0 and trial.number > 0:
        elapsed = time.time() - start_time
        print(f"\n✓ Trial {trial.number}/{n_trials} completed")
        print(f"  Current F1: {trial.value:.6f}")
        print(f"  Best F1: {study.best_value:.6f}")
        print(f"  Elapsed: {elapsed/60:.1f} min")
        est_remaining = ((elapsed / trial.number) * n_trials) - elapsed
        print(f"  Est. remaining: {est_remaining/60:.1f} min")

try:
    study.optimize(objective, n_trials=n_trials, timeout=timeout, 
                   callbacks=[callback], show_progress_bar=True)
except KeyboardInterrupt:
    print("\n⚠️  Interrupted by user")

end_time = time.time()
duration = end_time - start_time

print("\n" + "="*70)
print("OPTIMIZATION COMPLETE!")
print("="*70)
print(f"Time: {duration/60:.1f} min ({duration/3600:.2f} hours)")
print(f"Trials: {len(study.trials)}")
print(f"\n🏆 BEST F1 MACRO: {study.best_value:.6f}")
print("\n📊 Best hyperparameters:")
for param, value in study.best_params.items():
    print(f"  {param}: {value}")
print("="*70)

import pickle
with open('optuna_study.pkl', 'wb') as f:
    pickle.dump(study, f)
print("\n✅ Study saved to 'optuna_study.pkl'")

In [ ]:
# Optuna Visualizations
import optuna.visualization as vis

print("="*70)
print("OPTUNA OPTIMIZATION VISUALIZATIONS")
print("="*70)

# 1. Optimization History
fig1 = vis.plot_optimization_history(study)
fig1.update_layout(title="Optimization History: F1 Macro over Trials")
fig1.show()

# 2. Parameter Importance
fig2 = vis.plot_param_importances(study)
fig2.update_layout(title="Hyperparameter Importance")
fig2.show()

# 3. Parallel Coordinate
fig3 = vis.plot_parallel_coordinate(study)
fig3.update_layout(title="Parameter Relationships")
fig3.show()

# 4. Slice Plot
fig4 = vis.plot_slice(study)
fig4.update_layout(title="Individual Parameter Effects")
fig4.show()

# Top trials
print("\n" + "="*70)
print("TOP 10 TRIALS")
print("="*70)
trials_df = study.trials_dataframe().sort_values('value', ascending=False)
print(trials_df[['number', 'value', 'state']].head(10))

print("\nStatistics:")
print(f"  Best: {study.best_value:.6f}")
print(f"  Mean: {trials_df['value'].mean():.6f}")
print(f"  Std: {trials_df['value'].std():.6f}")
print("="*70)

In [ ]:
# Train Final Model with Best Parameters
print("="*70)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*70)

best_params = {
    'device': DEVICE if 'DEVICE' in globals() else 'cpu',
    'tree_method': 'hist',
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}
best_params.update(study.best_params)

print("\n📋 Final Parameters:")
for param, value in sorted(best_params.items()):
    print(f"  {param}: {value}")

# Verify with CV
print("\n🔄 Verifying with 5-Fold CV...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]
    
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_tr, y_tr, verbose=False)
    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores.append(f1)
    print(f"  Fold {fold + 1}: {f1:.6f}")

mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

print(f"\nCV F1: {mean_f1:.6f} (+/- {std_f1:.6f})")

# Train final model
print("\n🚀 Training on full data...")
model_best = xgb.XGBClassifier(**best_params)
model_best.fit(X_train, y_train, verbose=False)

y_test_pred = model_best.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred, average='macro')

print(f"\nTest F1: {test_f1:.6f}")

# Compare with baseline
baseline = 0.8002
print(f"\nBaseline: {baseline:.6f}")
print(f"Optuna:   {test_f1:.6f}")
print(f"Gain:     {test_f1 - baseline:+.6f} ({((test_f1/baseline)-1)*100:+.2f}%)")

if test_f1 > baseline:
    print("\n🎉 Hyperparameter tuning improved the model!")

# Save model
import pickle
with open('xgb_best_model.pkl', 'wb') as f:
    pickle.dump(model_best, f)
print("\n✅ Model saved to 'xgb_best_model.pkl'")
print("="*70)

In [57]:
%%time
xgb_clf = xgb.XGBClassifier(**xgb_better_params)
xgb_clf

CPU times: user 66 μs, sys: 2 μs, total: 68 μs
Wall time: 87.3 μs


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=1.0, colsample_bynode=1.0, colsample_bytree=0.8,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=0,
              max_depth=6, max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_class=3, num_parallel_tree=None, ...)

In [58]:
xgb_clf_fit = xgb_clf.fit(X_train, y_train)
xgb_clf_fit

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=1.0, colsample_bynode=1.0, colsample_bytree=0.8,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=0,
              max_depth=6, max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_class=3, num_parallel_tree=None, ...)

In [59]:
#xgb_preds = xgb_clf_fit.predict(test_dl.xs)
xgb_sc_preds = xgb_clf_fit.predict(X_test)

In [40]:
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

0.8002434487899772

In [53]:
#new trial with default params score
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

0.8002434487899772

In [60]:
#new trial with xgb better params score
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

0.8124452175684628

In [61]:
xgb_preds = xgb_clf_fit.predict(test_dl.xs)

In [62]:
xgb_preds.shape, test_df.shape

((2405,), (2405, 37))

In [63]:
sub_df

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium


In [66]:
!rm submission.csv
xgb_names = to.vocab[xgb_preds]
submit = sub_df
submit.Target = xgb_names
submit.to_csv('submission.csv',index=False)
submit

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium


In [46]:
!ls

 LICENSE
 SampleSubmission.csv
'Starter Notebook.ipynb'
 Test.csv
 Train.csv
 VariableDefinitions.csv
 dataorg-financial-health-prediction-challenge.zip
'dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n (1).zip:Zone.Identifier'
 financial-health-zindi-gbms.ipynb
 manifest-c388a32489998f91d25543edfbe8b78f20251204-19827-1nvty2o.json
 submission.csv
 submission_fastai.csv


## Adapt everything above

# F1 Score for Multiclass Classification

## Your Dataset Characteristics

**Target Classes:**
- Low: 6280 samples (65.3%)
- Medium: 2868 samples (29.8%)
- High: 470 samples (4.9%)

**Note**: Your dataset is imbalanced, especially the "High" class (only 4.9%). This makes F1 score the right choice over accuracy!

---

## F1 Score Averaging Methods

For multiclass problems, there are different ways to calculate F1:

### 1. **Macro F1** (Recommended for Competition)
```python
from sklearn.metrics import f1_score

# Calculate F1 for each class, then average (treats all classes equally)
f1_macro = f1_score(y_true, y_pred, average='macro')
```
- **Use when**: All classes are equally important (most competitions use this)
- Treats "High" class equally to "Low" class despite size difference
- **Most likely what your competition uses**

### 2. **Weighted F1**
```python
f1_weighted = f1_score(y_true, y_pred, average='weighted')
```
- **Use when**: Want to weight by class frequency
- Gives more importance to "Low" class (65%) than "High" (5%)

### 3. **Micro F1**
```python
f1_micro = f1_score(y_true, y_pred, average='micro')
```
- **Use when**: You care about overall performance
- Equals accuracy for multiclass problems

---

## Complete Implementation Example

### Using CatBoost (Recommended)

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from catboost import CatBoostClassifier, Pool
import pandas as pd

# Prepare data
X_train = train_df.drop('Target', axis=1)
y_train = train_df['Target']

# Identify categorical features
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

# Label encode target (High=0, Low=1, Medium=2 alphabetically)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Create validation split with stratification (important for imbalanced data!)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded  # Maintains class distribution
)

# Handle class imbalance with class weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}
print(f"Class weights: {class_weights_dict}")

# Create CatBoost model optimized for F1
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro',  # Use Macro F1 for evaluation
    class_weights=class_weights_dict,  # Handle imbalance
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)

# Create data pools
train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

# Train
model.fit(train_pool, eval_set=val_pool)

# Validate
val_pred = model.predict(val_pool).flatten()
val_pred_labels = le.inverse_transform(val_pred.astype(int))
val_true_labels = le.inverse_transform(y_val)

# Calculate F1 scores
f1_macro = f1_score(val_true_labels, val_pred_labels, average='macro')
f1_weighted = f1_score(val_true_labels, val_pred_labels, average='weighted')

print(f"\nValidation Macro F1: {f1_macro:.4f}")
print(f"Validation Weighted F1: {f1_weighted:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(val_true_labels, val_pred_labels))

# Predict on test
test_pool = Pool(test_df, cat_features=cat_features)
test_pred = model.predict(test_pool).flatten()
test_pred_labels = le.inverse_transform(test_pred.astype(int))

# Create submission
submission = pd.DataFrame({
    'ID': test_df.index,
    'Target': test_pred_labels
})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv")
```

---

## Using XGBoost with F1 Optimization

```python
import xgboost as xgb
from sklearn.metrics import f1_score

# Custom F1 evaluation function for XGBoost
def f1_eval_macro(preds, dtrain):
    labels = dtrain.get_label()
    # Reshape predictions for multiclass
    preds_reshaped = preds.reshape(len(np.unique(labels)), -1).T
    preds_class = np.argmax(preds_reshaped, axis=1)
    f1 = f1_score(labels, preds_class, average='macro')
    return 'f1_macro', f1

# Prepare data
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_encoded, test_size=0.2, random_state=42, stratify=y_train_encoded
)

# Handle categorical features
cat_cols = X_train.select_dtypes(include=['object']).columns
for col in cat_cols:
    le_col = LabelEncoder()
    X_tr[col] = le_col.fit_transform(X_tr[col].astype(str))
    X_val[col] = le_col.transform(X_val[col].astype(str))
    test_df[col] = le_col.transform(test_df[col].astype(str))

# Create DMatrix
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_val, label=y_val)

# Parameters
params = {
    'objective': 'multi:softmax',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'max_depth': 6,
    'eta': 0.05,
    'seed': 42
}

# Train with F1 monitoring
evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    feval=f1_eval_macro,
    early_stopping_rounds=50,
    verbose_eval=50
)

# Predict
dtest = xgb.DMatrix(test_df)
predictions = model.predict(dtest)
predicted_labels = le.inverse_transform(predictions.astype(int))
```

---

## Using LightGBM with F1 Optimization

```python
import lightgbm as lgb
from sklearn.metrics import f1_score

# Custom F1 metric for LightGBM
def lgb_f1_macro(y_true, y_pred):
    # y_pred is a 1D array of predicted probabilities for all classes
    y_pred = y_pred.reshape(len(np.unique(y_true)), -1).T
    y_pred_class = np.argmax(y_pred, axis=1)
    f1 = f1_score(y_true, y_pred_class, average='macro')
    return 'f1_macro', f1, True  # True means higher is better

# Prepare data (same encoding as XGBoost above)
# ... encode categorical features ...

# Create datasets
train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Parameters
params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'max_depth': 6,
    'seed': 42,
    'verbose': -1
}

# Train
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    feval=lgb_f1_macro,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)
```

---

## Cross-Validation with F1 Score

For more robust evaluation:

```python
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# Prepare data
X = train_df.drop('Target', axis=1)
y = le.fit_transform(train_df['Target'])

cat_features = X.select_dtypes(include=['object']).columns.tolist()

# 5-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold + 1}")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # Train model
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='MultiClass',
        eval_metric='TotalF1:average=Macro',
        cat_features=cat_features,
        random_seed=42,
        verbose=0
    )

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)

    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, verbose=False)

    # Predict and evaluate
    val_pred = model.predict(val_pool).flatten()
    f1 = f1_score(y_val, val_pred.astype(int), average='macro')
    f1_scores.append(f1)

    print(f"Fold {fold + 1} Macro F1: {f1:.4f}")

print(f"\nMean Macro F1: {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores):.4f})")
```

---

## Tips for Improving F1 Score

### 1. Handle Class Imbalance
```python
# Option A: Class weights (already shown above)
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)

# Option B: SMOTE (Synthetic Minority Over-sampling)
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_encoded, y_train)

# Option C: Adjust decision threshold (for probability predictions)
# Get probabilities instead of hard predictions
probs = model.predict_proba(val_pool)
# Adjust thresholds for each class to maximize F1
```

### 2. Focus on Minority Class (High)
```python
# Feature engineering specifically for rare class
# Analyze what makes "High" class different
high_samples = train_df[train_df['Target'] == 'High']
print(high_samples.describe())
```

### 3. Ensemble Models
```python
# Combine predictions from multiple models
from sklearn.ensemble import VotingClassifier

# Use soft voting (averages probabilities)
ensemble = VotingClassifier(
    estimators=[
        ('cat', catboost_model),
        ('xgb', xgboost_model),
        ('lgb', lightgbm_model)
    ],
    voting='soft'
)
```

---

## Quick Reference: F1 Score by Library

| Library | Metric Parameter | Value |
|---------|-----------------|-------|
| CatBoost | `eval_metric` | `'TotalF1:average=Macro'` |
| XGBoost | Custom `feval` | Use f1_score function |
| LightGBM | Custom `feval` | Use f1_score function |
| scikit-learn | `f1_score()` | `average='macro'` |

---

## What to Check in Competition Rules

Look for phrases like:
- "macro-averaged F1"
- "mean F1"
- "F1 score averaged across all classes"

This will tell you which averaging method to use!


In [ ]:
# ==========================================
# XGBoost Default Parameters (Baseline)
# ==========================================
# These are the default values for XGBoost multiclass classification
# Use this as your baseline before hyperparameter tuning

xgb_default_params = {
    # ==========================================
    # TASK PARAMETERS (Required for multiclass)
    # ==========================================
    'objective': 'multi:softprob',      # Multiclass with probability output
    'num_class': 3,                     # 3 classes: Low, Medium, High
    'eval_metric': 'mlogloss',          # Multiclass log loss
    
    # ==========================================
    # TREE STRUCTURE (Core parameters)
    # ==========================================
    'max_depth': 6,                     # DEFAULT: 6 - Maximum tree depth
    'min_child_weight': 1,              # DEFAULT: 1 - Min sum of instance weight in child
    'gamma': 0,                         # DEFAULT: 0 - Min loss reduction for split
    
    # ==========================================
    # LEARNING RATE & BOOSTING
    # ==========================================
    'learning_rate': 0.3,               # DEFAULT: 0.3 (eta) - Step size shrinkage
    'n_estimators': 100,                # DEFAULT: 100 - Number of boosting rounds
    
    # ==========================================
    # SAMPLING (Prevents overfitting)
    # ==========================================
    'subsample': 1.0,                   # DEFAULT: 1.0 - Row sampling ratio
    'colsample_bytree': 1.0,            # DEFAULT: 1.0 - Column sampling per tree
    'colsample_bylevel': 1.0,           # DEFAULT: 1.0 - Column sampling per level
    'colsample_bynode': 1.0,            # DEFAULT: 1.0 - Column sampling per node
    
    # ==========================================
    # REGULARIZATION
    # ==========================================
    'reg_alpha': 0,                     # DEFAULT: 0 - L1 regularization (Lasso)
    'reg_lambda': 1,                    # DEFAULT: 1 - L2 regularization (Ridge)
    
    # ==========================================
    # CLASS IMBALANCE (Your case: High=4.9%)
    # ==========================================
    'max_delta_step': 0,                # DEFAULT: 0 - Max output change (0=no limit)
    # Note: For multiclass imbalance, use sample_weight in fit() instead of scale_pos_weight
    
    # ==========================================
    # SYSTEM PARAMETERS
    # ==========================================
    'tree_method': 'hist',              # Use 'hist' for speed (default: 'auto')
    'random_state': 42,                 # For reproducibility (default: 0)
    'n_jobs': -1,                       # Use all CPU cores (default: None)
    'verbosity': 0,                     # Silent mode (default: 1)
}

# Print summary
print("="*70)
print("XGBoost Default Parameters Loaded")
print("="*70)
print(f"Total parameters: {len(xgb_default_params)}")
print(f"\nKey settings:")
print(f"  Objective: {xgb_default_params['objective']}")
print(f"  Classes: {xgb_default_params['num_class']}")
print(f"  Max depth: {xgb_default_params['max_depth']}")
print(f"  Learning rate: {xgb_default_params['learning_rate']}")
print(f"  N estimators: {xgb_default_params['n_estimators']}")
print(f"  Tree method: {xgb_default_params['tree_method']}")
print("="*70)

In [ ]:
# ==========================================
# BAYESIAN HYPERPARAMETER OPTIMIZATION
# Line-by-Line Explanation
# ==========================================

# Import necessary libraries
import numpy as np                                    # For numerical operations
import pandas as pd                                   # For data manipulation
import xgboost as xgb                                 # XGBoost classifier
from sklearn.model_selection import StratifiedKFold  # For stratified cross-validation
from sklearn.metrics import f1_score, make_scorer, classification_report  # For evaluation
from sklearn.utils.class_weight import compute_class_weight  # For handling imbalance
from skopt import BayesSearchCV                       # Bayesian optimization
from skopt.space import Real, Integer                 # Define search space types

# -----------------------------------------
# STEP 1: Prepare Training Data
# -----------------------------------------

# Extract features (X) and labels (y) from your preprocessed FastAI TabularPandas object
# to.train.xs = Features after Categorify + FillMissing + Normalize preprocessing
# to.train.ys = Target labels (still as DataFrame column)
X_train_full = to.train.xs                            # Shape: (7695, 42) - all training features
y_train_full = to.train.ys.values.ravel()             # Shape: (7695,) - class indices (0, 1, 2)
                                                      # .values converts DataFrame → numpy array
                                                      # .ravel() flattens 2D array [[0],[1],[2]] → 1D array [0,1,2]

# -----------------------------------------
# STEP 2: Calculate Class Weights
# -----------------------------------------

# Why? Your classes are imbalanced: Low=65%, Medium=30%, High=5%
# Without weights, model ignores rare "High" class

# compute_class_weight calculates balanced weights using formula:
#   weight[class] = total_samples / (n_classes × samples_in_class)
#
# Example calculation:
#   Low (class 0): 9618 / (3 × 6280) = 0.51
#   Medium (class 1): 9618 / (3 × 2868) = 1.12
#   High (class 2): 9618 / (3 × 470) = 6.82 ← Rare class gets highest weight!

class_weights = compute_class_weight(
    class_weight='balanced',                          # Use balanced weighting formula
    classes=np.unique(y_train_full),                  # [0, 1, 2] - all unique classes
    y=y_train_full                                    # The actual labels
)
# Result: class_weights = array([0.51, 1.12, 6.82])

print("="*60)
print("CLASS IMBALANCE ANALYSIS")
print("="*60)

# Count samples per class
unique, counts = np.unique(y_train_full, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count:,} samples ({count/len(y_train_full)*100:.1f}%)")

print(f"\nCalculated class weights:")
for cls, weight in enumerate(class_weights):
    print(f"  Class {cls}: {weight:.3f}x weight")

# Create sample_weights array: maps each sample to its class weight
# Example: if y_train_full[0] = 2 (High), then sample_weights[0] = 6.82
sample_weights = np.array([class_weights[int(i)] for i in y_train_full])
# Shape: (7695,) - one weight per training sample

# -----------------------------------------
# STEP 3: Define Hyperparameter Search Space
# -----------------------------------------

# BayesSearchCV needs to know the type and range of each hyperparameter
# Integer() = discrete whole numbers
# Real() = continuous floating point numbers
# prior='log-uniform' = sample more densely at small values (0.01, 0.03, 0.1, 0.3, 1, 3, 10)
# prior='uniform' = sample evenly across range

search_spaces = {
    # --- Tree Structure Parameters ---
    # Control how complex individual trees can be
    
    'max_depth': Integer(4, 10, name='max_depth'),
    # Maximum depth of each decision tree
    # Range: 4 to 10 levels deep
    # Lower (4) = simpler trees, less overfitting, faster training
    # Higher (10) = complex trees, can learn intricate patterns, slower
    # Will test values: 4, 5, 6, 7, 8, 9, 10
    
    'min_child_weight': Integer(1, 7, name='min_child_weight'),
    # Minimum sum of instance weights needed in a child node to allow split
    # Range: 1 to 7
    # Lower (1) = allows tiny leaf nodes, can overfit
    # Higher (7) = forces larger leaves, more conservative
    # With sample_weights, this becomes important for rare class handling
    
    'gamma': Real(0, 0.5, prior='uniform', name='gamma'),
    # Minimum loss reduction required to make a split
    # Range: 0.0 to 0.5
    # 0 = split whenever there's ANY improvement (greedy)
    # 0.5 = only split if loss reduces substantially (conservative)
    # Higher gamma = simpler trees
    
    # --- Learning Rate Parameters ---
    # Control how fast model learns and how many trees needed
    
    'learning_rate': Real(0.01, 0.3, prior='log-uniform', name='learning_rate'),
    # How much each tree contributes to final prediction
    # Range: 0.01 to 0.3, sampled logarithmically
    # Low (0.01) = slow learning, need many trees, better generalization
    # High (0.3) = fast learning, need fewer trees, risk overfitting
    # log-uniform ensures we test: 0.01, 0.02, 0.05, 0.1, 0.2, 0.3
    
    'n_estimators': Integer(300, 1500, name='n_estimators'),
    # Number of boosting rounds (trees to build)
    # Range: 300 to 1500 trees
    # More trees = more learning capacity
    # But: low learning_rate requires MORE trees
    # Bayesian will learn: "if lr=0.01, need ~1000 trees; if lr=0.2, need ~400"
    
    # --- Sampling Parameters ---
    # Introduce randomness to prevent overfitting
    
    'subsample': Real(0.6, 1.0, prior='uniform', name='subsample'),
    # Fraction of training samples to use for each tree
    # Range: 60% to 100%
    # 1.0 = use all 7695 training samples per tree
    # 0.8 = randomly sample 80% (6156 samples) per tree
    # Each tree sees different random subset → diversity → better ensemble
    
    'colsample_bytree': Real(0.6, 1.0, prior='uniform', name='colsample_bytree'),
    # Fraction of features to use when building each tree
    # Range: 60% to 100%
    # You have 42 features after preprocessing
    # 1.0 = each tree considers all 42 features
    # 0.8 = each tree randomly uses 34 features
    # Prevents over-reliance on strongest features
    
    # --- Regularization Parameters ---
    # Add penalties to prevent overfitting
    
    'reg_alpha': Real(1e-5, 10, prior='log-uniform', name='reg_alpha'),
    # L1 regularization (Lasso) - adds penalty: sum(|leaf_weights|)
    # Range: 0.00001 to 10, sampled logarithmically
    # Effect: Pushes some leaf weights to EXACTLY 0 (sparse model)
    # Good for: Feature selection, interpretability
    # log-uniform tests: 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10
    
    'reg_lambda': Real(1e-5, 100, prior='log-uniform', name='reg_lambda'),
    # L2 regularization (Ridge) - adds penalty: sum(leaf_weights²)
    # Range: 0.00001 to 100, sampled logarithmically
    # Effect: Shrinks all leaf weights toward 0 (smoother predictions)
    # Good for: General overfitting prevention
    
    # --- Class Imbalance Handling ---
    
    'max_delta_step': Integer(0, 5, name='max_delta_step'),
    # Maximum change allowed in leaf weight predictions
    # Range: 0 to 5
    # 0 = no limit (can predict extreme values for rare classes)
    # 5 = cap all leaf outputs between [-5, +5]
    # Useful for imbalanced data to prevent overconfident predictions on rare class
}

print(f"\n{'='*60}")
print("SEARCH SPACE CONFIGURATION")
print("="*60)
print(f"Total hyperparameters to optimize: {len(search_spaces)}")
print(f"\nSearch space:")
for param_name, param_space in search_spaces.items():
    print(f"  {param_name:25s}: {param_space}")

# -----------------------------------------
# STEP 4: Create Base XGBoost Model
# -----------------------------------------

# This defines the model type and fixed parameters
# Hyperparameters in search_spaces will override these defaults

xgb_model = xgb.XGBClassifier(
    # --- Task Parameters ---
    objective='multi:softprob',    # Multiclass classification with probability outputs
                                   # Returns prob matrix: [[P(Low), P(Med), P(High)], ...]
                                   
    num_class=3,                   # 3 target classes (Low=0, Medium=1, High=2)
    
    eval_metric='mlogloss',        # Multiclass log loss for monitoring
                                   # Lower is better: 0 = perfect, ~1.1 = random guessing
                                   
    # --- Performance Parameters ---
    tree_method='hist',            # Histogram-based tree building algorithm
                                   # Faster than 'exact' for your data size (9k samples)
                                   # Bins continuous features for efficient splitting
                                   
    random_state=42,               # Random seed for reproducibility
                                   # Same seed = same results every time
                                   
    n_jobs=-1,                     # Use all CPU cores for parallel training
                                   # -1 = detect and use all available cores
                                   
    verbosity=0                    # Suppress XGBoost training logs
                                   # 0 = silent, 1 = warnings, 2 = info
)

# -----------------------------------------
# STEP 5: Create F1 Macro Scorer
# -----------------------------------------

# BayesSearchCV needs a scoring function to optimize
# By default it knows 'accuracy', 'roc_auc', but not 'f1_macro'
# We create a custom scorer

def f1_macro_scorer(y_true, y_pred):
    """
    Calculate F1 Macro score for multiclass classification.
    
    Args:
        y_true: True class labels [0, 1, 2, 1, 0, ...] shape (n_samples,)
        y_pred: Predicted class labels [0, 1, 1, 1, 0, ...] shape (n_samples,)
    
    Returns:
        F1 macro score (float between 0 and 1, higher is better)
    
    How F1 Macro works:
        1. Calculate F1 for class 0 (Low): F1_0 = 0.85
        2. Calculate F1 for class 1 (Medium): F1_1 = 0.78
        3. Calculate F1 for class 2 (High): F1_2 = 0.62
        4. Average them: (0.85 + 0.78 + 0.62) / 3 = 0.75
    
    This treats all classes equally regardless of size!
    Rare "High" class (5%) has same weight as "Low" (65%)
    """
    return f1_score(
        y_true,           # Ground truth labels
        y_pred,           # Model predictions
        average='macro'   # Macro-averaging: simple mean of per-class F1 scores
    )

# Wrap function in make_scorer so sklearn recognizes it
f1_scorer = make_scorer(
    f1_macro_scorer,          # Our custom function
    greater_is_better=True    # Higher F1 = better (vs metrics like loss where lower = better)
)

# -----------------------------------------
# STEP 6: Set Up Stratified Cross-Validation
# -----------------------------------------

# Why Cross-Validation?
# Single train/test split can be misleading (maybe you got lucky/unlucky split)
# CV trains model K times on different data splits → more robust estimate

# Why STRATIFIED?
# Regular KFold might create folds with different class distributions
# Stratified maintains same 65:30:5 ratio in EVERY fold

cv = StratifiedKFold(
    n_splits=5,        # Create 5 folds
                       # Each fold: 80% train (6156 samples), 20% val (1539 samples)
                       
    shuffle=True,      # Randomly shuffle data before splitting
                       # Prevents bias if data is ordered by class
                       
    random_state=42    # Seed for reproducible splits
)

# How it works:
# Fold 1: Train on 80% → Validate on 20% → F1 = 0.82
# Fold 2: Train on different 80% → Validate on different 20% → F1 = 0.81
# Fold 3: Train on different 80% → Validate on different 20% → F1 = 0.83
# Fold 4: Train on different 80% → Validate on different 20% → F1 = 0.82
# Fold 5: Train on different 80% → Validate on different 20% → F1 = 0.81
# Average: (0.82 + 0.81 + 0.83 + 0.82 + 0.81) / 5 = 0.818

# -----------------------------------------
# STEP 7: Configure Bayesian Search
# -----------------------------------------

print(f"\n{'='*60}")
print("BAYESIAN OPTIMIZATION CONFIGURATION")
print("="*60)
print(f"Strategy: Intelligent hyperparameter search")
print(f"Iterations: 50 (each tries different hyperparameter combination)")
print(f"Cross-validation: 5-fold stratified")
print(f"Total model fits: 50 iterations × 5 folds = 250 models")
print(f"Optimization metric: F1 Macro Score")
print(f"Expected runtime: ~1-2 hours")
print("="*60)

# Create Bayesian search optimizer
bayes_search = BayesSearchCV(
    estimator=xgb_model,         # The base XGBoost model to tune
    
    search_spaces=search_spaces,  # The hyperparameter ranges defined above
    
    n_iter=50,                    # Number of hyperparameter combinations to try
                                  # More iterations = better results but slower
                                  # 50 is a good balance for production
    
    scoring=f1_scorer,            # Optimize for F1 Macro score
                                  # BayesSearchCV will try to MAXIMIZE this
    
    cv=cv,                        # Use our 5-fold stratified cross-validation
                                  # Each hyperparameter combo evaluated 5 times
    
    n_jobs=-1,                    # Parallelize across CV folds
                                  # Fit 5 folds simultaneously on different CPU cores
    
    verbose=2,                    # Print progress during search
                                  # 0 = silent, 1 = minimal, 2 = detailed progress
    
    random_state=42,              # Reproducible search
    
    return_train_score=True       # Also calculate training F1
                                  # Helps detect overfitting: if train_F1 >> test_F1
)

# -----------------------------------------
# STEP 8: Run Bayesian Optimization
# -----------------------------------------

print(f"\n🚀 Starting Bayesian Optimization...")
print(f"This will take approximately 1-2 hours.\n")

# Fit the Bayesian search
# This is where the magic happens!
bayes_search.fit(
    X_train_full,           # Training features (7695 samples × 42 features)
    y_train_full,           # Training labels (7695 class indices)
    sample_weight=sample_weights  # Class weights for each sample
                                  # Makes model pay attention to rare "High" class
)

# What happens inside .fit():
# 
# Iteration 1-5: Random exploration
#   Try random hyperparameter combinations to build initial understanding
#   Example: max_depth=6, lr=0.1, n_est=500 → F1=0.79
# 
# Iteration 6-50: Smart exploration + exploitation
#   Build Gaussian Process model of F1_score(hyperparameters)
#   Use acquisition function to pick next best hyperparameters:
#     - Exploitation: Try near known good regions (max_depth=7 worked, try 6 or 8)
#     - Exploration: Try unexplored regions (haven't tested high reg_alpha yet)
# 
# For EACH iteration:
#   1. Pick hyperparameters (smartly based on previous results)
#   2. Create XGBoost model with those hyperparameters
#   3. 5-fold CV:
#       Fold 1: Fit on 6156 samples, validate on 1539 → F1_1
#       Fold 2: Fit on 6156 samples, validate on 1539 → F1_2
#       Fold 3: Fit on 6156 samples, validate on 1539 → F1_3
#       Fold 4: Fit on 6156 samples, validate on 1539 → F1_4
#       Fold 5: Fit on 6156 samples, validate on 1539 → F1_5
#       Average: mean_F1 = (F1_1 + F1_2 + F1_3 + F1_4 + F1_5) / 5
#   4. Update Bayesian model with result
#   5. Repeat
# 
# After 50 iterations, select hyperparameters with highest mean_F1

print(f"\n✅ Bayesian Optimization Complete!\n")

# -----------------------------------------
# STEP 9: Analyze Results
# -----------------------------------------

print("="*60)
print("OPTIMIZATION RESULTS")
print("="*60)

# Best F1 score found across all iterations
print(f"\n🎯 Best CV F1 Macro Score: {bayes_search.best_score_:.6f}")

# Best hyperparameters that achieved this score
print(f"\n📊 Best Hyperparameters Found:")
for param, value in bayes_search.best_params_.items():
    # Format value based on type
    if isinstance(value, float):
        print(f"  {param:25s}: {value:.6f}")
    else:
        print(f"  {param:25s}: {value}")

# Get all results from cross-validation
cv_results = pd.DataFrame(bayes_search.cv_results_)

# Show top 5 hyperparameter configurations
print(f"\n{'='*60}")
print("TOP 5 HYPERPARAMETER CONFIGURATIONS")
print("="*60)

top_5 = cv_results.nlargest(5, 'mean_test_score')[
    ['mean_test_score', 'std_test_score', 'mean_train_score', 'params']
]

for idx, row in top_5.iterrows():
    rank = list(top_5.index).index(idx) + 1
    print(f"\nRank {rank}:")
    print(f"  Validation F1: {row['mean_test_score']:.6f} (±{row['std_test_score']:.6f})")
    print(f"  Training F1:   {row['mean_train_score']:.6f}")
    print(f"  Gap:           {row['mean_train_score'] - row['mean_test_score']:.6f}")
    print(f"  Parameters:    {row['params']}")

# Check for overfitting
best_train_score = cv_results.loc[bayes_search.best_index_, 'mean_train_score']
gap = best_train_score - bayes_search.best_score_

print(f"\n{'='*60}")
print("OVERFITTING ANALYSIS")
print("="*60)
print(f"Training F1:   {best_train_score:.6f}")
print(f"Validation F1: {bayes_search.best_score_:.6f}")
print(f"Gap:           {gap:.6f}")

if gap > 0.05:
    print("⚠️  WARNING: Possible overfitting detected (gap > 5%)")
    print("   Model performs significantly better on training than validation")
    print("   Consider: more regularization, less complex trees, more data")
else:
    print("✅ Good generalization - model not overfitting")

# -----------------------------------------
# STEP 10: Train Final Model on Full Data
# -----------------------------------------

print(f"\n{'='*60}")
print("TRAINING FINAL MODEL")
print("="*60)

# Get the best model from Bayesian search
# This model already has best hyperparameters set
best_model = bayes_search.best_estimator_

# During CV, each fold only used 80% of data for training
# Now retrain on ALL training data (100%) for maximum performance
print("Retraining best model on full training set (7,695 samples)...")

best_model.fit(
    X_train_full,          # All training features (no holdout)
    y_train_full,          # All training labels
    sample_weight=sample_weights  # Still use class weights
)

print("✅ Final model trained on full dataset")

# -----------------------------------------
# STEP 11: Validate on Held-Out Test Set
# -----------------------------------------

print(f"\n{'='*60}")
print("VALIDATION SET EVALUATION")
print("="*60)

# Your FastAI TabularPandas object has a separate validation set
# This is data the model has NEVER seen (not used in CV or final training)
X_val = to.valid.xs              # Validation features (1923 samples × 42 features)
y_val = to.valid.ys.values.ravel()  # Validation labels (1923 class indices)

# Make predictions on validation set
val_predictions = best_model.predict(X_val)
# Input: X_val shape (1923, 42)
# Output: val_predictions shape (1923,) with values [0, 1, 2]

# Calculate F1 score
val_f1 = f1_score(y_val, val_predictions, average='macro')

print(f"\n🎯 Validation F1 Macro Score: {val_f1:.6f}")
print(f"📈 Improvement over baseline (0.8002): {val_f1 - 0.8002:+.6f}")

if val_f1 > 0.8002:
    improvement_pct = ((val_f1 / 0.8002) - 1) * 100
    print(f"   ({improvement_pct:+.2f}% relative improvement)")

# Detailed per-class performance
print(f"\n{'='*60}")
print("DETAILED CLASSIFICATION REPORT (Validation Set)")
print("="*60)

# Map class indices back to names using your FastAI vocab
class_names = to.vocab  # ['High', 'Low', 'Medium'] or similar
print(classification_report(
    y_val,                # True labels: [0, 1, 2, ...]
    val_predictions,      # Predicted labels: [0, 1, 2, ...]
    target_names=[f"Class_{i} ({class_names[i]})" for i in range(len(class_names))],
    digits=4              # Show 4 decimal places
))

# This shows:
# - Precision: Of predicted "High", how many were actually High?
# - Recall: Of actual "High" samples, how many did we find?
# - F1-score: Harmonic mean of precision and recall
# - Support: Number of samples in each class

# -----------------------------------------
# STEP 12: Generate Test Predictions
# -----------------------------------------

print(f"{'='*60}")
print("GENERATING TEST SET PREDICTIONS")
print("="*60)

# Predict on competition test set (2,405 unseen samples)
test_predictions = best_model.predict(test_dl.xs)
# Input: test_dl.xs shape (2405, 42)
# Output: test_predictions shape (2405,) with values [0, 1, 2]

# Map numeric predictions to original class names
test_pred_labels = to.vocab[test_predictions]
# Example: [0, 1, 2, 0] → ['High', 'Low', 'Medium', 'High']

# Create submission DataFrame
submission = pd.DataFrame({
    'ID': test_df.index,          # Test sample IDs
    'Target': test_pred_labels    # Predicted class names
})

# Save to CSV for competition submission
submission.to_csv('submission_tuned_xgboost.csv', index=False)

print(f"✅ Predictions saved to: submission_tuned_xgboost.csv")
print(f"\nTest set prediction distribution:")
print(submission['Target'].value_counts())

print(f"\nExpected distribution (from training):")
train_dist = pd.Series(to.vocab[y_train_full]).value_counts()
print(train_dist)

print(f"\n{'='*60}")
print("🎉 HYPERPARAMETER TUNING COMPLETE!")
print("="*60)
print(f"\nNext steps:")
print(f"1. Submit 'submission_tuned_xgboost.csv' to competition")
print(f"2. Compare leaderboard score to baseline")
print(f"3. If score improves, consider:")
print(f"   - More Bayesian iterations (n_iter=100)")
print(f"   - Feature engineering")
print(f"   - Ensemble with other models (LightGBM, CatBoost)")
print("="*60)

In [ ]:
# Install required package for Bayesian Optimization
!pip install scikit-optimize

# XGBoost Hyperparameter Tuning with Bayesian Optimization

## 📚 What is Hyperparameter Tuning?

Hyperparameters are settings you configure BEFORE training (like `max_depth=6`, `learning_rate=0.1`). Unlike model parameters (weights learned during training), hyperparameters control HOW the model learns.

**Goal**: Find the hyperparameter combination that maximizes F1 Macro Score on validation data.

---

## 🔍 Why Bayesian Optimization?

### The Problem Space
You have 10 hyperparameters to tune. If each has 5 possible values:
- **Grid Search**: Tests all 5^10 = 9,765,625 combinations ❌ (would take years!)
- **Random Search**: Tests 100 random combinations 🤷 (decent but wasteful)
- **Bayesian Optimization**: Tests 50 SMART combinations ✅ (learns from each trial)

### How Bayesian Optimization Works

```
Trial 1-5: Random exploration
  → max_depth=4, lr=0.1 → F1=0.78
  → max_depth=8, lr=0.05 → F1=0.82 ← Good!
  
Trial 6: Bayesian model thinks "depth=8 worked well, let me try nearby values"
  → max_depth=7, lr=0.05 → F1=0.83 ← Even better!
  
Trial 7: "Let me also explore unexplored regions"
  → max_depth=5, lr=0.2 → F1=0.75 ← Not good
  
Trial 8: "Back to the good region with variations"
  → max_depth=8, lr=0.03 → F1=0.84 ← Best so far!
  
... continues intelligently balancing exploration vs exploitation
```

**Result**: Finds near-optimal hyperparameters in 50 trials instead of 10,000+

---

## 🎯 Why F1 Macro for Your Problem?

### Your Class Distribution
- **Low**: 6,280 samples (65.3%)
- **Medium**: 2,868 samples (29.8%)
- **High**: 470 samples (4.9%) ← Rare class!

### Accuracy vs F1 Macro

**If you used Accuracy:**
```python
# Model that ALWAYS predicts "Low"
Accuracy = 65.3%  # Decent score!
F1 for High class = 0.0  # Terrible! Never predicts High

This model is useless but has good accuracy!
```

**With F1 Macro:**
```python
F1_Low = 0.90
F1_Medium = 0.70
F1_High = 0.40  ← Penalizes poor performance on rare class

F1_Macro = (0.90 + 0.70 + 0.40) / 3 = 0.67
```

**Key Point**: F1 Macro treats all classes equally, forcing the model to perform well on the rare "High" class too.

---

## ⚖️ Handling Class Imbalance with Sample Weights

### The Problem
Without adjustment, the model sees:
- 6,280 "Low" examples → learns Low patterns very well
- 470 "High" examples → ignores High patterns (too few to matter)

### The Solution: Class Weights
```
Calculate balanced weights:
  weight_High = total_samples / (num_classes × count_High)
  weight_High = 9618 / (3 × 470) = 6.82

Each "High" sample now counts as 6.82 regular samples!

Low sample: weight = 0.51
Medium sample: weight = 1.12  
High sample: weight = 6.82 ← Makes model pay attention!
```

---

## 📊 Stratified K-Fold Cross-Validation

### Why Cross-Validation?
Single train/test split can be lucky or unlucky. CV gives robust estimate.

### Why STRATIFIED?
Regular K-Fold might create folds like:
```
Fold 1: 70% Low, 25% Medium, 5% High  ← Good distribution
Fold 2: 80% Low, 18% Medium, 2% High  ← Broken! Too few High
Fold 3: 60% Low, 35% Medium, 5% High  ← Different distribution
```

Stratified K-Fold maintains class distribution in EVERY fold:
```
Fold 1: 65.3% Low, 29.8% Medium, 4.9% High ← Matches original
Fold 2: 65.3% Low, 29.8% Medium, 4.9% High ← Matches original
Fold 3: 65.3% Low, 29.8% Medium, 4.9% High ← Matches original
```

**Result**: Every fold is representative → more reliable F1 score

---

## 🌳 Key XGBoost Hyperparameters Being Tuned

### 1. Tree Structure (Controls Model Complexity)

**`max_depth`** [4-10]: Maximum tree depth
```
max_depth=3:           max_depth=8:
   [root]                  [root]
   /    \                 /      \
 [L]    [R]            [node]  [node]
                        /  \    /  \
Shallow = Simple      ... many more levels ...
Good for small data   Deep = Complex patterns
                      Needs lots of data
```
- **Too low**: Underfitting (misses patterns)
- **Too high**: Overfitting (memorizes noise)
- **Your range [4-10]**: Let Bayesian find sweet spot

**`min_child_weight`** [1-7]: Minimum data points in leaf
```
min_child_weight=1:  Allows tiny leaves with 1 sample
  → Can overfit to noise
  
min_child_weight=5:  Forces ≥5 samples per leaf
  → More conservative, better generalization
```

**`gamma`** [0-0.5]: Minimum loss reduction to split
```
gamma=0: Split if any improvement (greedy)
gamma=0.3: Split only if substantial improvement (conservative)
```
- Higher gamma = simpler trees = less overfitting

---

### 2. Learning Control

**`learning_rate`** [0.01-0.3]: How much each tree contributes
```
Prediction = base + lr×tree1 + lr×tree2 + ...

lr=0.3 (high):
  tree1 adds 30% of its output → faster learning
  Needs fewer trees (300-500)
  Risk: overfitting
  
lr=0.01 (low):
  tree1 adds 1% of its output → slower learning
  Needs many trees (1000-2000)
  Benefit: better generalization
```

**`n_estimators`** [300-1500]: Number of trees
- More trees = more capacity to learn
- But with low learning_rate, you need LOTS of trees
- Early stopping prevents overfitting

---

### 3. Randomness (Prevents Overfitting)

**`subsample`** [0.6-1.0]: Row sampling per tree
```
subsample=1.0: Each tree sees all 9,618 samples
subsample=0.8: Each tree sees random 80% (7,694 samples)

Tree 1: trains on samples [1,2,3,5,7,9,10,...]
Tree 2: trains on samples [1,3,4,6,8,10,11,...] ← Different!
```
- Creates diversity in ensemble
- Like bagging in Random Forest

**`colsample_bytree`** [0.6-1.0]: Column sampling per tree
```
You have 42 features

colsample_bytree=0.8: Each tree uses random 80% (34 features)

Tree 1: uses [owner_age, income, expenses, ...]
Tree 2: uses [country, income, turnover, ...]  ← Different features!
```
- Prevents over-reliance on any single feature
- Especially good for correlated features

---

### 4. Regularization (Prevents Overfitting)

**`reg_alpha`** [0.00001-10]: L1 regularization (Lasso)
```
Adds penalty: sum(|leaf_weights|)

Effect: Pushes some leaf weights to EXACTLY 0
  → Sparse model (feature selection)
  → Simpler, more interpretable
```

**`reg_lambda`** [0.00001-100]: L2 regularization (Ridge)
```
Adds penalty: sum(leaf_weights²)

Effect: Shrinks all leaf weights toward 0
  → Smoother predictions
  → Better generalization
```

**Why log-uniform prior?**
```
Linear scale: [0.001, 0.01, 0.1, 1, 10]
  → Only tests 0.001 once
  
Log-uniform: [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]
  → More samples in small values (where differences matter)
```

---

### 5. Imbalance Handling

**`max_delta_step`** [0-5]: Caps leaf value changes
```
Without constraint (=0):
  Leaf for "High" class: output = +15.3 (extreme!)
  → Model becomes overconfident on rare class
  
With max_delta_step=2:
  Leaf output capped at +2.0
  → More conservative, stable predictions
```
- Useful for severe class imbalance
- Prevents model from making wild predictions on rare class

---

## 🔄 The Complete Workflow

### Step-by-Step Process

```
┌─────────────────────────────────────────────┐
│ 1. Load Data (9,618 samples, 42 features)  │
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 2. Calculate Class Weights                  │
│    Low: 0.51x   Medium: 1.12x   High: 6.82x│
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 3. Define Search Space (10 hyperparameters) │
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 4. Bayesian Optimization (50 iterations)    │
│                                             │
│  For each iteration:                        │
│    ┌─────────────────────────────────────┐ │
│    │ a. Pick hyperparameters smartly     │ │
│    └─────────────┬───────────────────────┘ │
│                  │                          │
│    ┌─────────────▼───────────────────────┐ │
│    │ b. 5-Fold Stratified Cross-Val      │ │
│    │                                      │ │
│    │    Fold 1: Train → Validate → F1    │ │
│    │    Fold 2: Train → Validate → F1    │ │
│    │    Fold 3: Train → Validate → F1    │ │
│    │    Fold 4: Train → Validate → F1    │ │
│    │    Fold 5: Train → Validate → F1    │ │
│    │                                      │ │
│    │    Average F1 = 0.823               │ │
│    └─────────────┬───────────────────────┘ │
│                  │                          │
│    ┌─────────────▼───────────────────────┐ │
│    │ c. Update Bayesian model            │ │
│    │    "depth=8, lr=0.05 was good,      │ │
│    │     next try depth=7, lr=0.04"      │ │
│    └─────────────────────────────────────┘ │
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 5. Select Best Hyperparameters              │
│    (Highest average F1 score)               │
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 6. Retrain on FULL training data            │
│    (No CV split, use all 9,618 samples)     │
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 7. Validate on held-out validation set      │
│    (Your to.valid data)                     │
└─────────────────────┬───────────────────────┘
                      │
┌─────────────────────▼───────────────────────┐
│ 8. Predict on test set (2,405 samples)      │
│    Generate submission file                 │
└─────────────────────────────────────────────┘
```

---

## 📈 Expected Results

### Baseline (Your Current Model)
- Validation F1 Macro: **0.8002**
- Using default XGBoost hyperparameters

### After Bayesian Optimization
- Expected F1 Macro: **0.82-0.85** (+2-5% improvement)
- Why improvement:
  1. ✅ Optimized tree complexity for your data size
  2. ✅ Proper class imbalance handling
  3. ✅ Regularization tuned to prevent overfitting
  4. ✅ Learning rate + n_estimators balanced
  
### What to Watch For

**Good Signs** ✅
```
Train F1: 0.870
Test F1:  0.845
Gap:      0.025 (< 5%)  → Good generalization!
```

**Warning Signs** ⚠️
```
Train F1: 0.950
Test F1:  0.810
Gap:      0.140 (14%)  → Overfitting! Need more regularization
```

---

## ⏱️ Runtime Expectations

```
50 iterations × 5 CV folds × ~20 seconds per fold
= 250 model fits × 20 seconds
≈ 5,000 seconds
≈ 1.4 hours

Actual time depends on:
- CPU cores (more cores = faster parallel CV)
- Your data size (9,618 samples = medium, manageable)
- n_estimators in search space (more trees = slower)
```

**Progress Updates:**
```
Iteration 1/50: Best F1=0.7823
Iteration 10/50: Best F1=0.8145  ← Improving!
Iteration 25/50: Best F1=0.8267  ← Still improving
Iteration 40/50: Best F1=0.8289  ← Converging
Iteration 50/50: Best F1=0.8291  ← Final best
```

---

## 🚀 Ready to Run!

The next code cell contains the complete implementation. It will:
1. ✅ Handle your class imbalance automatically
2. ✅ Optimize for F1 Macro (your competition metric)
3. ✅ Use stratified CV for reliable estimates
4. ✅ Print detailed progress and results
5. ✅ Generate a submission file

**Before running:**
```python
# Install required package (if not already installed)
!pip install scikit-optimize
```

**After running, you'll get:**
- Best hyperparameters found
- Validation F1 score
- Detailed classification report
- `submission_tuned_xgboost.csv` file ready for submission

Let's find those optimal hyperparameters! 🎯